In [41]:
# !pip install requests

import requests
from xml.etree import ElementTree as ET
import json
import pprint
import time
import aiohttp
import asyncio

In [28]:
#asked ChatGPT how to tweak the API call so it is useable in python

# find 1000 alz articles 
base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
params = {
    "db": "pubmed",
    "term": "Alzheimers AND 2024[pdat]",
    "retmax": "1000",
    "retmode": "xml"
}

# Send request
response = requests.get(base_url, params=params)
root = ElementTree.fromstring(response.text)

# Extract article IDs
alz_ids = [id_elem.text for id_elem in root.findall(".//Id")]
print(f"Found {len(alz_ids)} Alzheimers articles.")
print(alz_ids[:10])  # show first 10 IDs

Found 1000 Alzheimers articles.
['41058862', '41024939', '40980211', '40973408', '40973407', '40973404', '40973402', '40973401', '40973397', '40970099']


In [ ]:
# found all 1000 cancer papers

base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
params = {
    "db": "pubmed", # from PubMed
    "term": "cancer AND 2024[pdat]", #cancer
    "retmax": "1000", # only send 1000 articles
    "retmode": "xml" # sends in xml format
}

response = requests.get(base_url, params=params)
root = ElementTree.fromstring(response.text)

cancer_ids = [id_elem.text for id_elem in root.findall(".//Id")]
print(f"Found {len(cancer_ids)} cancer articles.")
print(cancer_ids[:10])  # show first 10 IDs


Found 1000 cancer articles.
['41146951', '41144996', '41142171', '41142170', '41140641', '41140424', '41140420', '41140418', '41134009', '41111483']


In [57]:
# asked ChatGPT how to now get metadata for each article ID fetched earlier
# used ChatGPT to add the time.sleep portion, add article id title being pulled for metadata

# fetch metadata for the 1000 alzheimers
fetch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
all_alz_metadata = []
failed_pmids = []
alz_pmids =[]

for i in range(0, len(alz_ids), 200):  # batches of 200
    batch_ids = ",".join(alz_ids[i:i+200])
    fetch_params = {
        "db": "pubmed",
        "id": batch_ids,
        "retmode": "xml"
    }
    try:
        fetch_response = requests.get(fetch_url, params=fetch_params, timeout=30)
        fetch_response.raise_for_status()
        root = ET.fromstring(fetch_response.text)
    except Exception as e:
        print(f"Error fetching batch {i//200+1}: {e}")
        continue

    for article in root:
        try:
            title_elem = article.find(".//ArticleTitle")
            abstract_elems = article.findall(".//Abstract/AbstractText")
            abstract_text = " ".join("".join(elem.itertext()) for elem in abstract_elems) if abstract_elems else None
            journal_elem = article.find(".//Journal/Title")
            pmid_elem = article.find(".//PMID")
            date_elem = article.find(".//PubDate/Year")

            metadata = {
                "PMID": pmid_elem.text if pmid_elem is not None else None,
                "ArticleTitle": title_elem.text if title_elem is not None else None,
                "AbstractText": abstract_text,
                "Journal": journal_elem.text if journal_elem is not None else None,
                "YearPublished": date_elem.text if date_elem is not None else None
            }
            all_alz_metadata.append(metadata)
            alz_ids.append(metadata['PMID'])
        except Exception as e:
            pmid_text = pmid_elem.text if pmid_elem is not None else "UNKNOWN"
            print(f"Error parsing article PMID {pmid_text}: {e}")
            failed_pmids.append(pmid_text)
            continue

    time.sleep(1)

print(f"Retrieved metadata for {len(all_alz_metadata)} Alzheimers articles.")
print(f"Failed PMIDs in first pass: {len(failed_pmids)}")


Retrieved metadata for 1000 Alzheimers articles.
Failed PMIDs in first pass: 0


In [56]:
# asked ChatGPT how to now get metadata for each article ID fetched earlier
# used ChatGPT to add the time.sleep portion, add article id title being pulled for metadata, and to make it more robust - was skipping the 1 cancer article that is a book

# fetch metadata for the 1000 cancer articles
fetch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
all_cancer_metadata = []
failed_pmids = []
cancer_pmids = []

for i in range(0, len(cancer_ids), 200):  # batches of 200
    batch_ids = ",".join(cancer_ids[i:i+200])
    fetch_params = {
        "db": "pubmed",
        "id": batch_ids,
        "retmode": "xml"
    }
    try:
        fetch_response = requests.get(fetch_url, params=fetch_params, timeout=30)
        fetch_response.raise_for_status()
        root = ET.fromstring(fetch_response.text)
    except Exception as e:
        print(f"Error fetching batch {i//200+1}: {e}")
        continue

    for article in root:
        try:
            title_elem = article.find(".//ArticleTitle")
            abstract_elems = article.findall(".//Abstract/AbstractText")
            abstract_text = " ".join("".join(elem.itertext()) for elem in abstract_elems) if abstract_elems else None
            journal_elem = article.find(".//Journal/Title")
            pmid_elem = article.find(".//PMID")
            date_elem = article.find(".//PubDate/Year")

            metadata = {
                "PMID": pmid_elem.text if pmid_elem is not None else None,
                "ArticleTitle": title_elem.text if title_elem is not None else None,
                "AbstractText": abstract_text,
                "Journal": journal_elem.text if journal_elem is not None else None,
                "YearPublished": date_elem.text if date_elem is not None else None
            }
            all_cancer_metadata.append(metadata)
            cancer_pmids.append(metadata['PMID'])

        except Exception as e:
            pmid_text = pmid_elem.text if pmid_elem is not None else "UNKNOWN"
            print(f"Error parsing article PMID {pmid_text}: {e}")
            failed_pmids.append(pmid_text)
            continue

    time.sleep(1)

print(f"Retrieved metadata for {len(all_cancer_metadata)} cancer articles.")
print(f"Failed PMIDs in first pass: {len(failed_pmids)}")


Retrieved metadata for 1000 cancer articles.
Failed PMIDs in first pass: 0


In [52]:
json_alz_metadata = json.dumps(all_alz_metadata, indent=2)
print(json_alz_metadata[:500])

[
  {
    "PMID": "41058862",
    "ArticleTitle": "Deep learning assessment of disproportionately enlarged subarachnoid-space hydrocephalus in Hakim's disease or idiopathic normal pressure hydrocephalus.",
    "AbstractText": "Disproportionately enlarged subarachnoid-space hydrocephalus (DESH) is a key feature of Hakim's disease (synonymous with idiopathic normal pressure hydrocephalus; iNPH). However, it previously had been only subjectively evaluated. This study aims to evaluate the usefulness


In [53]:
json_cancer_metadata = json.dumps(all_cancer_metadata, indent=2)
print(json_cancer_metadata[:500])

[
  {
    "PMID": "41146951",
    "ArticleTitle": "CDK4/6 inhibitors display a class effect in inducing differentiation of neuroblastoma cells.",
    "AbstractText": "Neuroblastoma is the most common extracranial solid tumour in infants and children, accounting for approximately 15% of paediatric cancer mortality. These tumours are unique in that a subset, namely stage MS, frequently undergo spontaneous regression or differentiation. Differentiation therapy, where cancer cells are re-routed back


In [58]:
duplicates = set(cancer_pmids) & set(alz_pmids)
print(f"🔍 Found {len(duplicates)} PMIDs that appear in both datasets.")
print(duplicates)

🔍 Found 0 PMIDs that appear in both datasets.
set()
